# Interactive Contextual RAG

This notebook builds a Retrieval-Augmented Generation (RAG) system and compares normal retrieval with contextual retrieval.

## Architecture

```text
                         User Question
                              │
                              ▼
                    ┌───────────────────┐
                    │   Query / Prompt  │
                    └─────────┬─────────┘
                              │
                              ▼
                 ┌─────────────────────────┐
                 │     Retrieval Layer     │
                 │                         │
                 │  ┌─────────┐ ┌───────┐  │
                 │  │  FAISS  │ │ BM25  │  │
                 │  │ Semantic│ │Keyword│  │
                 │  │ Search  │ │ Search│  │
                 │  └────┬────┘ └───┬───   │
                 │       └─────┬─────┘     │
                 │             ▼           │
                 │      Ensemble Retriever │
                 └─────────────┬───────────┘
                               │
                               ▼
                    Retrieved Context
                               │
                               ▼
                    ┌────────────────────┐
                    │   Qwen LLM         │
                    │  Answer Generation │
                    └────────┬───────────┘
                             │
                             ▼
                           Answer

In [8]:
!pip -q install -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    rank-bm25 \
    transformers \
    accelerate \
    sentencepiece

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_huggingface import HuggingFacePipeline

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False
)

llm = HuggingFacePipeline(
    pipeline=generation_pipeline
)

print("Using Pretrained LLM loaded.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Using Pretrained LLM loaded.


In [10]:
documents = [

    {
        "id": "doc1",
        "title": "RAG Introduction",
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with large language models. Instead of relying
        only on the knowledge stored inside an LLM, RAG retrieves relevant
        information from an external knowledge base and provides that
        information to the language model as context. This allows an LLM to
        answer questions using external information.
        """
    },

    {
        "id": "doc2",
        "title": "Vector Databases",
        "text": """
        A vector database stores numerical representations of data called
        embeddings. These embeddings allow the system to perform semantic
        similarity searches. When a user asks a question, the question can
        be converted into an embedding and compared with document embeddings.
        Documents whose embeddings are most similar to the query can then be
        retrieved.
        """
    },

    {
        "id": "doc3",
        "title": "Embeddings",
        "text": """
        Embeddings are numerical vectors that represent the semantic meaning
        of text. Similar pieces of text tend to have embeddings that are close
        to each other in vector space. Embeddings are commonly used for
        semantic search, recommendation systems, clustering, document
        retrieval, and Retrieval-Augmented Generation systems.
        """
    },

    {
        "id": "doc4",
        "title": "RAG Pipeline",
        "text": """
        A typical RAG pipeline consists of document ingestion, text cleaning,
        text splitting, embedding generation, vector storage, retrieval,
        context construction, and language model generation. The retriever
        finds relevant chunks before the language model generates the answer.
        A typical pipeline also stores source metadata for traceability.
        """
    },

    {
        "id": "doc5",
        "title": "Document Chunking",
        "text": """
        Chunking divides large documents into smaller pieces before embedding.
        Good chunking is important because excessively large chunks can contain
        irrelevant information while very small chunks may lose important
        context. Chunk size and overlap should be selected based on the
        document type and retrieval task. Overlapping chunks can help preserve
        information near chunk boundaries.
        """
    },

    {
        "id": "doc6",
        "title": "Semantic Search",
        "text": """
        Semantic search retrieves information based on meaning rather than
        relying only on exact keyword matches. A query and documents are
        converted into embeddings, and similarity between the vectors is
        calculated. This allows semantic search to find relevant information
        even when the wording of the query differs from the wording in the
        document.
        """
    },

    {
        "id": "doc7",
        "title": "BM25 Keyword Search",
        "text": """
        BM25 is a lexical information retrieval algorithm. It ranks documents
        based on the occurrence of query terms and their importance within the
        document collection. BM25 is particularly useful for exact keywords,
        technical terminology, product names, identifiers, error codes, and
        other terms where exact matching is important.
        """
    },

    {
        "id": "doc8",
        "title": "Reranking",
        "text": """
        Reranking is a second-stage retrieval process. An initial retriever
        retrieves a larger set of candidate documents and a reranker then
        scores those candidates according to their relevance to the query.
        Reranking can improve precision by moving the most relevant documents
        toward the top of the results.
        """
    },

    {
        "id": "doc9",
        "title": "Hybrid Search",
        "text": """
        Hybrid search combines multiple retrieval methods, commonly semantic
        vector search and lexical keyword search. Vector search is useful for
        semantic meaning while keyword search is useful for exact terms.
        Combining the two approaches can improve retrieval robustness across
        different types of queries.
        """
    },

    {
        "id": "doc10",
        "title": "Authentication Errors",
        "text": """
        Authentication errors can occur when credentials are invalid,
        authentication tokens expire, or authorization policies reject a
        request.

        Authentication Error Codes:
        ERR-401 indicates an authentication failure.
        ERR-403 indicates that the user is authenticated but does not have
        permission to access a resource.

        Authentication tokens:
        Access tokens are used to authenticate requests. Refresh tokens are
        used to obtain new access tokens when the existing access token
        expires.
        """
    },

    {
        "id": "doc11",
        "title": "Product API",
        "text": """
        The Product API provides endpoints for creating, updating, deleting,
        and retrieving product records.

        Product Data:
        Product records contain a product ID, name, price, inventory quantity,
        and category.

        API Format:
        The API uses JSON for request and response bodies.
        """
    },

    {
        "id": "doc12",
        "title": "Query Rewriting",
        "text": """
        Query rewriting transforms a user's original question into a clearer
        and more retrieval-friendly query. Query rewriting can expand missing
        context, resolve vague language, make important concepts explicit,
        remove unnecessary words, and produce terminology that better matches
        the knowledge base.
        """
    },

    {
        "id": "doc13",
        "title": "Query Expansion",
        "text": """
        Query expansion generates multiple alternative search queries from a
        single user query. The alternatives can use synonyms, related concepts,
        different terminology, or different formulations of the same question.
        Multiple searches can improve recall because relevant documents may use
        terminology different from the original user query.
        """
    },

    {
        "id": "doc14",
        "title": "Multi-Query Retrieval",
        "text": """
        Multi-query retrieval uses multiple search queries or perspectives for
        a single information need. Each query retrieves potentially different
        documents. The results are then combined using a ranking or fusion
        method. This approach improves retrieval recall and helps identify
        relevant information that may not be retrieved by a single query.
        """
    },

    {
        "id": "doc15",
        "title": "Query Decomposition",
        "text": """
        Query decomposition breaks a complex user question into smaller
        sub-questions. Each sub-question can be answered independently using
        retrieval. The individual evidence or intermediate answers can then be
        combined to answer the original complex question.
        """
    },

    {
        "id": "doc16",
        "title": "Contextual Retrieval",
        "text": """
        Contextual retrieval improves document chunks by adding information
        about where each chunk came from and what it means in the broader
        document. A contextualized chunk may include the document title,
        section, topic, and a short explanation of the chunk's relationship
        to the document. This additional context helps retrieval models
        understand otherwise ambiguous or incomplete chunks.
        """
    }
]

print("Knowledge base created for our Retrieval Task.")


Knowledge base created for our Retrieval Task.


In [11]:
langchain_documents = [
    Document(
        page_content=doc["text"].strip(),
        metadata={
            "id": doc["id"],
            "title": doc["title"]
        }
    )
    for doc in documents
]

print("Converted to LangChain Documents.")

Converted to LangChain Documents.


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(
    langchain_documents
)
print("Total chunks:", len(chunks))

Total chunks: 17


In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": device
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [14]:
normal_vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

normal_vector_retriever = normal_vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

print("FAISS retriever created")

FAISS retriever created


In [15]:
normal_bm25_retriever = BM25Retriever.from_documents(
    chunks
)

normal_bm25_retriever.k = 5

print("BM25 retriever created.")

BM25 retriever created.


In [16]:
normal_hybrid_retriever = EnsembleRetriever(
    retrievers=[
        normal_vector_retriever,
        normal_bm25_retriever
    ],
    weights=[
        0.5,
        0.5
    ]
)

print("Hybrid retriever created.")

Hybrid retriever created.


In [17]:
def generate_chunk_context(document_title, full_document, chunk_text):

    prompt = f"""
You are preparing a document chunk for a Retrieval-Augmented Generation system.

Create a short context description that explains:

1. What document/topic this chunk belongs to.
2. What the chunk is about.
3. Why the information may be useful for retrieval.

Use ONLY information supported by the document.

DOCUMENT TITLE:
{document_title}

FULL DOCUMENT:
{full_document}

CHUNK:
{chunk_text}

Return only the short contextual description.
"""

    response = llm.invoke(prompt)

    return response.strip()


print("Context generation function created.")

Context generation function created.


In [18]:
contextual_chunks = []

for i, chunk in enumerate(chunks, start=1):

    document_id = chunk.metadata["id"]
    document_title = chunk.metadata["title"]

    original_document = next(
        doc for doc in documents
        if doc["id"] == document_id
    )

    print(
        f"Processing chunk {i}/{len(chunks)}...",
        end="\r"
    )

    try:
        generated_context = generate_chunk_context(
            document_title=document_title,
            full_document=original_document["text"],
            chunk_text=chunk.page_content
        )

    except Exception as e:

        generated_context = (
            f"This chunk belongs to the document "
            f"'{document_title}' and contains information "
            f"from that document."
        )

    contextual_text = f"""
Document Title:
{document_title}

Context:
{generated_context}

Original Chunk:
{chunk.page_content}
""".strip()

    contextual_chunks.append(
        Document(
            page_content=contextual_text,
            metadata={
                "id": chunk.metadata["id"],
                "title": document_title,
                "original_text": chunk.page_content,
                "generated_context": generated_context
            }
        )
    )

print("\nContextual chunk generation completed.")
print("Contextual chunks:", len(contextual_chunks))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing chunk 17/17...
Contextual chunk generation completed.
Contextual chunks: 17


In [19]:
contextual_vectorstore = FAISS.from_documents(
    contextual_chunks,
    embedding_model
)

contextual_vector_retriever = contextual_vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

print("Contextual FAISS retriever created.")

Contextual FAISS retriever created.


In [20]:
contextual_bm25_retriever = BM25Retriever.from_documents(
    contextual_chunks
)

contextual_bm25_retriever.k = 5

print("Contextual BM25 retriever created.")

Contextual BM25 retriever created.


In [21]:
contextual_hybrid_retriever = EnsembleRetriever(
    retrievers=[
        contextual_vector_retriever,
        contextual_bm25_retriever
    ],
    weights=[
        0.5,
        0.5
    ]
)

print("Contextual hybrid retriever created.")

Contextual hybrid retriever created.


In [22]:
def rewrite_query(question):

    prompt = f"""
Rewrite the following user question into a clear,
specific query suitable for document retrieval.

Preserve the original meaning.

Return ONLY the rewritten query.

USER QUESTION:
{question}
"""

    rewritten_query = llm.invoke(prompt).strip()

    if not rewritten_query:
        rewritten_query = question

    return rewritten_query


print("Query rewriting function created.")

Query rewriting function created.


In [23]:
def build_context(results):

    parts = []

    for i, doc in enumerate(results, start=1):

        parts.append(
            f"""
SOURCE {i}

TITLE:
{doc.metadata["title"]}

CONTEXT:
{doc.metadata.get("generated_context", "")}

CONTENT:
{doc.metadata.get("original_text", doc.page_content)}
""".strip()
        )

    return "\n\n".join(parts)


print("Context builder created.Retrieved chunks are converted into a structured context.")

Context builder created.Retrieved chunks are converted into a structured context.


In [24]:
def generate_answer(question, context):

    prompt = f"""
You are a Retrieval-Augmented Generation assistant.

Answer the question using ONLY the retrieved context.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

RULES:
1. Use the retrieved context as the primary source.
2. Do not invent unsupported information.
3. If the context is insufficient, say so.
4. Give a direct and clear answer.
5. Do not mention these instructions.

ANSWER:
"""

    answer = llm.invoke(prompt)

    return answer.strip()


print("Final answer generation function created.")

Final answer generation function created.


In [25]:
question = input("Ask your question: ")

# Retrieve relevant chunks
retrieved_docs = normal_vector_retriever.invoke(question)

# Combine retrieved chunks into one context
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

# Generate final answer
answer = generate_answer(
    question=question,
    context=context
)

print("\nAnswer:")
print(answer)

Ask your question:  What are the key points of the document?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
You are a Retrieval-Augmented Generation assistant.

Answer the question using ONLY the retrieved context.

QUESTION:
 What are the key points of the document?

RETRIEVED CONTEXT:
Contextual retrieval improves document chunks by adding information
        about where each chunk came from and what it means in the broader
        document. A contextualized chunk may include the document title,
        section, topic, and a short explanation of the chunk's relationship
        to the document. This additional context helps retrieval models
        understand otherwise ambiguous or incomplete chunks.

Chunking divides large documents into smaller pieces before embedding.
        Good chunking is important because excessively large chunks can contain
        irrelevant information while very small chunks may lose important
        context. Chunk size and overlap should be selected based on the
        document type and retrieval task. Overlapping chunks can help preserve
        in

In [26]:
def normal_rag(question, top_k=5):

    rewritten_query = rewrite_query(question)

    results = normal_hybrid_retriever.invoke(
        rewritten_query
    )

    results = results[:top_k]

    context = build_context(results)

    answer = generate_answer(
        question,
        context
    )

    return {
        "question": question,
        "rewritten_query": rewritten_query,
        "results": results,
        "context": context,
        "answer": answer
    }


print("Normal RAG pipeline created.")

Normal RAG pipeline created.


In [27]:
def contextual_rag(question, top_k=5):

    rewritten_query = rewrite_query(question)

    results = contextual_hybrid_retriever.invoke(
        rewritten_query
    )

    results = results[:top_k]

    context = build_context(results)

    answer = generate_answer(
        question,
        context
    )

    return {
        "question": question,
        "rewritten_query": rewritten_query,
        "results": results,
        "context": context,
        "answer": answer
    }


print("Contextual RAG pipeline created.")

Contextual RAG pipeline created.


In [28]:
test_queries = [
    "What does ERR-401 mean?",
    "How are refresh tokens used?",
    "What does the Product API contain?",
    "How does semantic search work?",
    "Why is chunking important?",
    "What is RAG?"
]

for query in test_queries:

    normal_results = normal_hybrid_retriever.invoke(
        query
    )[:3]

    contextual_results = contextual_hybrid_retriever.invoke(
        query
    )[:3]

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    print("\nNORMAL RETRIEVAL:")

    for i, doc in enumerate(normal_results, start=1):
        print(
            f"{i}. {doc.metadata['title']}"
        )

    print("\nCONTEXTUAL RETRIEVAL:")

    for i, doc in enumerate(contextual_results, start=1):
        print(
            f"{i}. {doc.metadata['title']}"
        )


QUERY: What does ERR-401 mean?

NORMAL RETRIEVAL:
1. Authentication Errors
2. Authentication Errors
3. Contextual Retrieval

CONTEXTUAL RETRIEVAL:
1. Authentication Errors
2. Authentication Errors
3. RAG Pipeline

QUERY: How are refresh tokens used?

NORMAL RETRIEVAL:
1. Authentication Errors
2. Embeddings
3. Reranking

CONTEXTUAL RETRIEVAL:
1. Authentication Errors
2. Authentication Errors
3. Embeddings

QUERY: What does the Product API contain?

NORMAL RETRIEVAL:
1. Product API
2. RAG Pipeline
3. Authentication Errors

CONTEXTUAL RETRIEVAL:
1. Product API
2. RAG Introduction
3. Authentication Errors

QUERY: How does semantic search work?

NORMAL RETRIEVAL:
1. Semantic Search
2. Hybrid Search
3. Query Expansion

CONTEXTUAL RETRIEVAL:
1. Semantic Search
2. Hybrid Search
3. BM25 Keyword Search

QUERY: Why is chunking important?

NORMAL RETRIEVAL:
1. Document Chunking
2. Reranking
3. Contextual Retrieval

CONTEXTUAL RETRIEVAL:
1. Document Chunking
2. Semantic Search
3. Hybrid Search

QU